Perform feature selection and retrain models by using Recursive Feature Elimination (RFE), permutation importance, and principal component analysis (PCA) to reduce dimensionality while maintaining predictive accuracy.

In [1]:
import joblib
import os
import shap
import numpy as np

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
save_folder = "task31_saved_files"

saved_models = "saved_models"
preprocessor = joblib.load(os.path.join(save_folder, "preprocessor.pkl"))
feature_config = joblib.load(os.path.join(save_folder, "feature_config.pkl"))
x_test = joblib.load(os.path.join(save_folder, "x_test.pkl"))
y_test = joblib.load(os.path.join(save_folder, "y_test.pkl"))
x_train = joblib.load(os.path.join(save_folder, "x_train.pkl"))
y_train = joblib.load(os.path.join(save_folder, "y_train.pkl"))

x_train_processed = preprocessor.transform(x_train)
x_test_processed = preprocessor.transform(x_test)

baseline_xgb_model = joblib.load(os.path.join(saved_models, "baseline_xgb_model.pkl"))

tuned_xgb_model = joblib.load(os.path.join(saved_models, "tuned_xgb_model.pkl"))

print("All saved files loaded successfully!")

All saved files loaded successfully!


In [4]:
import numpy as np
import pandas as pd

print("Training Data Shape:", x_train_processed.shape)
print("Testing Data Shape:", x_test_processed.shape)

print("\nData Type:", type(x_train_processed))

print("\nTraining Data Sparse:",hasattr(x_train_processed, "toarray"))

feature_names = preprocessor.get_feature_names_out()

print("\nNumber of Features:", len(feature_names))

print("\nTarget Distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest Target Distribution:")
print(pd.Series(y_test).value_counts())

Training Data Shape: (18786, 3049)
Testing Data Shape: (4697, 3049)

Data Type: <class 'scipy.sparse._csr.csr_matrix'>

Training Data Sparse: True

Number of Features: 3049

Target Distribution:
recommend_flag
1    15448
0     3338
Name: count, dtype: int64

Test Target Distribution:
recommend_flag
1    3863
0     834
Name: count, dtype: int64


Recursive Feature Elimination (RFE)

In [8]:
from sklearn.feature_selection import RFE
from sklearn.svm import LinearSVC

In [12]:
# RFE estimator
rfe_estimator = LinearSVC(
    C=1.0,
    class_weight='balanced',
    dual='auto',
    random_state=42,
    max_iter=3000
)

# RFE
rfe_selector = RFE(
    estimator=rfe_estimator,
    n_features_to_select=100,
    step=0.1,
    verbose=1
)

rfe_selector.fit(x_train_processed, y_train)

x_train_rfe = rfe_selector.transform(x_train_processed)
x_test_rfe = rfe_selector.transform(x_test_processed)

selected_features = np.array(feature_names)[rfe_selector.support_]

print("\nOriginal Feature Count:", len(feature_names))
print("Selected Feature Count:", len(selected_features))

print("\nRFE Training Shape:", x_train_rfe.shape)
print("RFE Testing Shape:", x_test_rfe.shape)

print("\nTop Selected Features:")
print(selected_features[:30])


Fitting estimator with 3049 features.
Fitting estimator with 2745 features.
Fitting estimator with 2441 features.
Fitting estimator with 2137 features.
Fitting estimator with 1833 features.
Fitting estimator with 1529 features.
Fitting estimator with 1225 features.
Fitting estimator with 921 features.
Fitting estimator with 617 features.
Fitting estimator with 313 features.

Original Feature Count: 3049
Selected Feature Count: 100

RFE Training Shape: (18786, 100)
RFE Testing Shape: (4697, 100)

Top Selected Features:
['text__115' 'text__117' 'text__140' 'text__2nd' 'text__2p' 'text__32'
 'text__accents' 'text__achieve' 'text__actual' 'text__additionally'
 'text__adds' 'text__adore' 'text__aesthetic' 'text__age' 'text__ankles'
 'text__apparently' 'text__appear' 'text__attached' 'text__basic'
 'text__became' 'text__bring' 'text__ca' 'text__choice' 'text__classic'
 'text__classy' 'text__cleaning' 'text__complaint' 'text__count'
 'text__crops' 'text__crotch']


Retrain XGBoost using RFE features

In [14]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [16]:
# RFE XGBoost Model
rfe_xgb_model = XGBClassifier(
    colsample_bytree=0.5335242593390661,
    gamma=3.5631475509938375,
    learning_rate=0.022983153291559925,
    max_depth=4,
    min_child_weight=10,
    n_estimators=491,
    subsample=0.6956174435600316,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

rfe_xgb_model.fit(x_train_rfe, y_train)

y_pred_rfe = rfe_xgb_model.predict(x_test_rfe)
y_prob_rfe = rfe_xgb_model.predict_proba(x_test_rfe)[:, 1]

# Evaluation
print("RFE XGBoost Performance")
print("-----------------------")

print("Accuracy:", accuracy_score(y_test, y_pred_rfe))
print("Precision:", precision_score(y_test, y_pred_rfe))
print("Recall:", recall_score(y_test, y_pred_rfe))
print("F1 Score:", f1_score(y_test, y_pred_rfe))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rfe))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rfe))

RFE XGBoost Performance
-----------------------
Accuracy: 0.8394720034064297
Precision: 0.8461367178802048
Recall: 0.983691431529899
F1 Score: 0.9097438352884846
ROC-AUC: 0.6798013310811356

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.17      0.28       834
           1       0.85      0.98      0.91      3863

    accuracy                           0.84      4697
   macro avg       0.77      0.58      0.59      4697
weighted avg       0.82      0.84      0.80      4697



Conclusion: RFE reduced the features from 3,049 to 100 (96.72% reduction). However, the XGBoost model achieved only 83.95% accuracy and 67.98% ROC-AUC, showing a significant performance drop compared with the original model. Therefore, RFE reduced dimensionality but failed to maintain predictive accuracy.

Permutation Importance

In [17]:
from sklearn.model_selection import train_test_split

In [18]:
x_train_pi, x_val_pi, y_train_pi, y_val_pi = train_test_split(x_train_processed, y_train, test_size=0.2, random_state=42, stratify=y_train) 
print("Training Shape:", x_train_pi.shape)
print("Validation Shape:", x_val_pi.shape)

Training Shape: (15028, 3049)
Validation Shape: (3758, 3049)


In [20]:
pi_model = XGBClassifier(
    colsample_bytree=0.5335242593390661,
    gamma=3.5631475509938375,
    learning_rate=0.022983153291559925,
    max_depth=4,
    min_child_weight=10,
    n_estimators=491,
    subsample=0.6956174435600316,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

pi_model.fit(x_train_pi, y_train_pi)

print("Permutation Importance model trained successfully!")

Permutation Importance model trained successfully!


In [21]:
from sklearn.inspection import permutation_importance

In [24]:
import numpy as np
from sklearn.inspection import permutation_importance

# Select 1000 validation samples
x_val_sample = x_val_pi[:1000].toarray()
y_val_sample = np.asarray(y_val_pi)[:1000]

perm_result = permutation_importance(
    pi_model,
    x_val_sample,
    y_val_sample,
    scoring="f1",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

print("Permutation Importance calculated successfully!")

Permutation Importance calculated successfully!


In [25]:
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": perm_result.importances_mean,
    "Std": perm_result.importances_std
})

importance_df = importance_df.sort_values(by="Importance", ascending=False)

print("Top 20 Important Features:")
display(importance_df.head(20))

Top 20 Important Features:


,Feature,Importance,Std
1,num__rating,0.096583,0.003250
2899,text__was,0.003439,0.000439
2355,text__size,0.002960,0.000830
2,num__vader_negative,0.001827,0.000926
1435,text__it,0.001381,0.001408
1900,text__perfect,0.001299,0.000235
2419,text__soft,0.001225,0.000748
2890,text__wanted,0.001164,0.000378
1909,text__petite,0.001050,0.000793
2955,text__wide,0.000697,0.000249


Select Top 100 features

In [28]:
# Select top 100 features
top_100_features = importance_df.head(100)["Feature"].values

# Get their original column indices
top_100_indices = [
    np.where(feature_names == feature)[0][0]
    for feature in top_100_features
]

# Transform training and testing data
x_train_pi_selected = x_train_processed[:, top_100_indices]
x_test_pi_selected = x_test_processed[:, top_100_indices]

print("Original Features:", x_train_processed.shape[1])
print("Selected Features:", len(top_100_indices))
print("Training Shape:", x_train_pi_selected.shape)
print("Testing Shape:", x_test_pi_selected.shape)

Original Features: 3049
Selected Features: 100
Training Shape: (18786, 100)
Testing Shape: (4697, 100)


In [30]:
pi_xgb_model = XGBClassifier(
    colsample_bytree=0.5335242593390661,
    gamma=3.5631475509938375,
    learning_rate=0.022983153291559925,
    max_depth=4,
    min_child_weight=10,
    n_estimators=491,
    subsample=0.6956174435600316,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

pi_xgb_model.fit(x_train_pi_selected, y_train)

y_pred_pi = pi_xgb_model.predict(x_test_pi_selected)
y_prob_pi = pi_xgb_model.predict_proba(x_test_pi_selected)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_pi))
print("Precision:", precision_score(y_test, y_pred_pi))
print("Recall:", recall_score(y_test, y_pred_pi))
print("F1 Score:", f1_score(y_test, y_pred_pi))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_pi))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_pi))

Accuracy: 0.9406003832233341
Precision: 0.9678851174934726
Recall: 0.9596168780740357
F1 Score: 0.9637332640062395
ROC-AUC: 0.9790133722687914

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.85      0.84       834
           1       0.97      0.96      0.96      3863

    accuracy                           0.94      4697
   macro avg       0.89      0.91      0.90      4697
weighted avg       0.94      0.94      0.94      4697



Conclusion – Permutation Importance
Permutation Importance reduced the features from 3,049 to 100 (96.72% reduction). The retrained XGBoost model achieved 94.06% accuracy, 96.37% F1-score, and 97.90% ROC-AUC. Its performance remained close to the original Bayesian XGBoost model, showing that Permutation Importance reduced dimensionality while maintaining good predictive performance.

Apply TruncatedSVD

In [34]:
from sklearn.decomposition import TruncatedSVD

In [36]:
svd = TruncatedSVD(n_components=100, random_state=42)

# Fit only on training data
x_train_svd = svd.fit_transform(x_train_processed)

# Transform test data
x_test_svd = svd.transform(x_test_processed)

print("Original Features:", x_train_processed.shape[1])
print("Reduced Components:", x_train_svd.shape[1])
print("Training Shape:", x_train_svd.shape)
print("Testing Shape:", x_test_svd.shape)

print("Total Explained Variance Ratio:",
      svd.explained_variance_ratio_.sum())


Original Features: 3049
Reduced Components: 100
Training Shape: (18786, 100)
Testing Shape: (4697, 100)
Total Explained Variance Ratio: 0.9453334536531868


Train XGBoost on reduced data

In [39]:
svd_xgb_model = XGBClassifier(
    colsample_bytree=0.5335242593390661,
    gamma=3.5631475509938375,
    learning_rate=0.022983153291559925,
    max_depth=4,
    min_child_weight=10,
    n_estimators=491,
    subsample=0.6956174435600316,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

svd_xgb_model.fit(x_train_svd, y_train)

# Predictions
y_pred_svd = svd_xgb_model.predict(x_test_svd)
y_prob_svd = svd_xgb_model.predict_proba(x_test_svd)[:, 1]

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred_svd))
print("Precision:", precision_score(y_test, y_pred_svd))
print("Recall:", recall_score(y_test, y_pred_svd))
print("F1 Score:", f1_score(y_test, y_pred_svd))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_svd))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svd))


Accuracy: 0.9429423035980413
Precision: 0.9672472056147647
Recall: 0.9632410044007248
F1 Score: 0.9652399481193256
ROC-AUC: 0.9789818675735052

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.85      0.84       834
           1       0.97      0.96      0.97      3863

    accuracy                           0.94      4697
   macro avg       0.90      0.91      0.90      4697
weighted avg       0.94      0.94      0.94      4697



### Conclusion – Feature Selection and Dimensionality Reduction

In this task, three techniques — Recursive Feature Elimination (RFE), Permutation Importance, and TruncatedSVD — were implemented to reduce the original 3,049 features to 100 features or components.

Among the three techniques, RFE achieved 83.95% accuracy, while Permutation Importance achieved 94.06% and TruncatedSVD achieved the highest accuracy of 94.29%. TruncatedSVD also retained 94.53% explained variance.

Overall, Permutation Importance and TruncatedSVD significantly reduced dimensionality while maintaining predictive performance close to the original Bayesian XGBoost model. Therefore, both techniques proved more effective than RFE for this dataset.